In [21]:
#Be sure to run these pip installs before running this script:
#pip install requests
#pip install boto3

import requests
import json
import os
import urllib3
import boto3
from datetime import datetime
import matplotlib.pyplot as plt
import io
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

In [23]:
s3_client = boto3.client('s3')

bucket_name = "s3-bucket-project1-3601"

In [24]:
# NCDOT traffic counts
import urllib


def lambda_function(event, context):

     #api_key = os.environ['TOMTOM_API_KEY']

     #latitude = ["a", "b"] # fill in with the found values
     #longitude = ["a", "b"]
     ids = ["0600000063", "0600000145", "0600000829", "0600000107", "0600000108", "0600000414", "0600000412", "0600000044", "0600000065", "0600001182"]

     http = urllib3.PoolManager()
     url = f"https://arcgis.com"

     for i in range(len(ids)):


        params = {
            "where": "ID = '" + ids[i] + "'",
            "outFields": "Location_ID,ROUTE,LOCATION,AADT_2023,AADT_2022",
            "f": "json" 
        }

        encoded_params = urllib.parse.urlencode(params)
        full_url = f"{url}?{encoded_params}"

        try:
            response = http.request('GET', full_url)
            if response.status_code==200:
                    data = json.loads(response.data.decode('utf-8'))
                    #flow_data = data.get("flowSegmentData", {})

                    now = datetime.utcnow()
                    timestamp_str = now.isoformat()

                   # traffic_record = {
                    #     "StationID": "", # Fill in later
                    #     "Timestamp": timestamp_str,
                    #     "current_speed": int(flow_data.get("currentSpeed",0)),
                    #     "free_flow_speed": int(flow_data.get("freeFlowSpeed",0)),
                    #     "current_delay": int(flow_data.get("currentDelay",0)),
                    #     "confidence": int(flow_data.get("confidence",0)) # for flags of when we shoudl somehow retrain?
                          
                    # }

                    s3_key = f"data/arcgis_export/year={now.year}/month={now.strftime('%m')}/day={now.strftime('%d')}/traffic_{now.strftime('%H%M%S')}.json"

                    s3_client.put_object(
                        Bucket=bucket_name,
                        Key=s3_key,
                        Body=json.dumps(data),
                        ContentType='application/json'
                    )

                    print(f"Successfully uploaded data to s3://{bucket_name}/{s3_key}")
                    return{"statusCode": 200, "body": "Data uploaded to s3 successfully!"}
            return {"statusCode": response.status, "body": "Failure!"}
        except Exception as e:
             print(f"Error encountered: {e}")
             return {"statusCode": 201, "body": str(e)}

            


In [25]:
# TomTom speed API
def lambda_function(event, context):

     api_key = os.environ['TOMTOM_API_KEY']

     latitude = ["a", "b"] # fill in with the found values
     longitude = ["a", "b"]

     http = urllib3.PoolManager()

     for i in range(len(latitude)):
        url = f"https://tomtom.com{api_key}&point={latitude[i]},{longitude[i]}&unit=MPH"
        try:
            response = http.request('GET', url)
            if response.status_code==200:
                    data = json.loads(response.data.decode('utf-8'))
                    flow_data = data.get("flowSegmentData", {})

                    now = datetime.utcnow()
                    timestamp_str = now.isoformat()

                    traffic_record = {
                        "StationID": "", # Fill in later
                        "Timestamp": timestamp_str,
                        "current_speed": int(flow_data.get("currentSpeed",0)),
                        "free_flow_speed": int(flow_data.get("freeFlowSpeed",0)),
                        "current_delay": int(flow_data.get("currentDelay",0)),
                        "confidence": int(flow_data.get("confidence",0)) # for flags of when we shoudl somehow retrain?
                          
                    }

                    s3_key = f"raw_traffic/year={now.year}/month={now.strftime('%m')}/day={now.strftime('%d')}/traffic_{now.strftime('%H%M%S')}.json"

                    s3_client.put_object(
                        Bucket=bucket_name,
                        Key=s3_key,
                        Body=json.dumps(traffic_record),
                        ContentType='application/json'
                    )

                    print(f"Successfully uploaded data to s3://{bucket_name}/{s3_key}")
                    return{"statusCode": 200, "body": "Data uploaded to s3 successfully!"}
            return {"statusCode": response.status, "body": "Failure!"}
        except Exception as e:
             print(f"Error encountered: {e}")
             return {"statusCode": 201, "body": str(e)}

            
def get_current_speed(latitude, longitude, api_key):
    url = f"https://tomtom.com"

    params = {
        'key': api_key,
        'point': f"{latitude},{longitude}",
        'unit': "MPH",
        "openLr": "false"
    }

    response = requests.get(url, params)

    if response.status_code==200:
        data = response.json()
        flow_data = data.get("flowSegmentData", {})

        data_we_want = {
            "current_speed": flow_data.get("currentSpeed"),
            "free_flow_speed": flow_data.get("freeFlowSpeed"),
            "current_delay": flow_data.get("currentDelay"),
            "confidence": flow_data.get("confidence") # for flags of when we shoudl somehow retrain?
        }

        return data_we_want
    else:
        print(f"Error in retrieving data: {response.status_code}")